# exo_toolkit Pipeline Demo

End-to-end walkthrough: **Fetch → Clean → Search → Vet → Score → Classify**

**Target**: TOI-700 (TIC 150428135) — a nearby M-dwarf hosting three confirmed planets, including TOI-700 d in the habitable zone (Gilbert et al. 2020).  
**Mission**: TESS  
**Purpose**: Demonstrate the full pipeline on a real, well-characterised system and produce a human-readable candidate report.

---

> **Disclaimer**: This pipeline identifies *candidate signals* only. It never claims confirmed planet status. All outputs are for reproducibility and community discussion purposes.

## 0. Imports and Configuration

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

# Ensure src/ is on the path when running outside an installed package
import os
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if os.path.join(repo_root, 'src') not in sys.path:
    sys.path.insert(0, os.path.join(repo_root, 'src'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from exo_toolkit.fetch import fetch_lightcurve
from exo_toolkit.clean import clean_lightcurve
from exo_toolkit.search import search_lightcurve
from exo_toolkit.vet import vet_signal
from exo_toolkit.scoring import score_candidate
from exo_toolkit.pathway import classify_submission_pathway

# Target configuration
TARGET_ID   = 'TIC 150428135'   # TOI-700
MISSION     = 'TESS'
SECTOR      = None               # None = all available sectors

# Known stellar parameters from TIC / Stassun et al. 2019
STELLAR_RADIUS_RSUN = 0.420
STELLAR_MASS_MSUN   = 0.416

print(f'Target  : {TARGET_ID}')
print(f'Mission : {MISSION}')
print(f'R_star  : {STELLAR_RADIUS_RSUN} R_sun')
print(f'M_star  : {STELLAR_MASS_MSUN} M_sun')

---
## 1. Fetch

`fetch_lightcurve` queries the MAST archive via Lightkurve and returns a `FetchResult` with provenance metadata.

In [ ]:
fetch_result = fetch_lightcurve(TARGET_ID, mission=MISSION)
lc_raw = fetch_result.light_curve
prov = fetch_result.provenance

print(f'Cadence        : {prov.cadence_seconds} s')
print(f'Sectors        : {prov.sectors}')
print(f'Pipeline       : {prov.pipeline}')
print(f'Fetched at     : {prov.fetched_at}')
print(f'N cadences     : {len(lc_raw.time)}')

---
## 2. Clean

`clean_lightcurve` removes NaN cadences, applies sigma clipping, normalises the flux to unit median, and applies a windowed Savitzky-Golay detrend to suppress long-term systematics.

In [ ]:
clean_result = clean_lightcurve(lc_raw)
lc = clean_result.light_curve
cprov = clean_result.provenance

print(f'Cadences raw     : {cprov.n_cadences_raw}')
print(f'Cadences cleaned : {cprov.n_cadences_cleaned}')
print(f'Removed          : {cprov.n_cadences_raw - cprov.n_cadences_cleaned}')
print(f'Sigma-clip sigma : {cprov.sigma_clip_sigma}')
print(f'Window length    : {cprov.window_length} cadences')

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=False)

# Raw
axes[0].scatter(np.asarray(lc_raw.time.jd), np.asarray(lc_raw.flux.value),
                s=0.5, color='steelblue', alpha=0.6, rasterized=True)
axes[0].set_ylabel('Raw flux (e$^{-}$/s)', fontsize=10)
axes[0].set_title(f'{TARGET_ID} — Raw PDCSAP flux', fontsize=11)

# Cleaned
axes[1].scatter(np.asarray(lc.time.jd), np.asarray(lc.flux.value),
                s=0.5, color='darkorange', alpha=0.6, rasterized=True)
axes[1].set_ylabel('Normalised flux', fontsize=10)
axes[1].set_xlabel('Time (BJD)', fontsize=10)
axes[1].set_title('Cleaned and normalised', fontsize=11)

plt.tight_layout()
plt.savefig('figures/01_raw_vs_cleaned.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: figures/01_raw_vs_cleaned.png')

---
## 3. Search

`search_lightcurve` runs a Box Least Squares (BLS) periodicity search over a grid of trial periods and durations. Iterative transit masking recovers multiple signals. Returns a list of `CandidateSignal` objects sorted by SDE (Signal Detection Efficiency).

In [ ]:
signals = search_lightcurve(lc)

print(f'Signals found: {len(signals)}')
print()
print(f'{"#":<3} {"Period (d)":<12} {"Epoch (BJD)":<16} {"Duration (hr)":<14} {"Depth (ppm)":<12} {"SNR":<8} {"Transit count"}')
print('-' * 80)
for i, s in enumerate(signals):
    print(f'{i:<3} {s.period_days:<12.4f} {s.epoch_bjd:<16.4f} {s.duration_hours:<14.2f} {s.depth_ppm:<12.0f} {s.snr:<8.1f} {s.transit_count}')

---
## 4. Vet

`vet_signal` computes per-transit depths, odd/even comparison, secondary eclipse SNR, transit shape (ingress/egress fraction), and data-gap fraction directly from the light curve arrays. Stellar and catalog diagnostics are supplied as keyword arguments.

In [ ]:
if not signals:
    raise RuntimeError('No signals found — cannot continue.')

signal = signals[0]
print(f'Vetting signal: P = {signal.period_days:.4f} d, depth = {signal.depth_ppm:.0f} ppm, SNR = {signal.snr:.1f}')

vet_result = vet_signal(
    lc,
    signal,
    stellar_radius_rsun=STELLAR_RADIUS_RSUN,
    stellar_mass_msun=STELLAR_MASS_MSUN,
    contamination_ratio=0.01,       # TIC contamination ratio
    centroid_offset_sigma=None,     # not computed (no TPF)
    quality_flag_fraction=0.02,
)

diag = vet_result.diagnostics
feat = vet_result.features

print()
print('--- Raw Diagnostics ---')
print(f'Individual transit depths   : {[f"{d:.4f}" for d in diag.individual_depths] if diag.individual_depths else "N/A"}')
print(f'Depth odd  (ppm)            : {diag.depth_odd_ppm}')
print(f'Depth even (ppm)            : {diag.depth_even_ppm}')
print(f'Secondary SNR               : {diag.secondary_snr}')
print(f'Ingress/egress fraction     : {diag.ingress_egress_fraction}')
print(f'Data gap fraction           : {diag.data_gap_fraction}')

print()
print('--- Extracted Features ---')
for field, val in feat.model_fields.items():
    v = getattr(feat, field)
    if v is not None:
        print(f'  {field:<40}: {v:.4f}')

---
## 5. Score

`score_candidate` computes the Bayesian log-score posterior over six hypotheses and derives FPP, detection confidence, novelty score, habitability interest, follow-up value, and submission readiness.

In [ ]:
posterior, scores = score_candidate(signal, feat)

print('--- Hypothesis Posterior ---')
hyp_labels = [
    ('planet_candidate',            posterior.planet_candidate),
    ('eclipsing_binary',            posterior.eclipsing_binary),
    ('background_eclipsing_binary', posterior.background_eclipsing_binary),
    ('stellar_variability',         posterior.stellar_variability),
    ('instrumental_artifact',       posterior.instrumental_artifact),
    ('known_object',                posterior.known_object),
]
for name, prob in hyp_labels:
    bar = '█' * int(prob * 40)
    print(f'  {name:<35} {prob:.4f}  {bar}')

print()
print('--- Derived Scores ---')
print(f'  False positive probability : {scores.false_positive_probability:.4f}')
print(f'  Detection confidence       : {scores.detection_confidence:.4f}')
print(f'  Novelty score              : {scores.novelty_score:.4f}')
print(f'  Habitability interest      : {scores.habitability_interest:.4f}')
print(f'  Follow-up value            : {scores.followup_value:.4f}')
print(f'  Submission readiness       : {scores.submission_readiness:.4f}')

---
## 6. Classify

`classify_submission_pathway` applies an ordered decision gate to route the candidate to the appropriate submission pathway.

In [ ]:
pathway = classify_submission_pathway(signal, feat, posterior, scores)

pathway_descriptions = {
    'known_object_annotation':      'Matches a known catalog object.',
    'tfop_ready':                   'Strong TESS candidate; suitable for TFOP follow-up.',
    'planet_hunters_discussion':    'Interesting but ambiguous; recommend community review.',
    'kepler_archive_candidate':     'Strong Kepler/K2 archival candidate.',
    'github_only_reproducibility':  'Low-confidence or exploratory; internal record only.',
    'paper_or_preprint_candidate':  'Exceptionally strong; suitable for formal publication.',
}

print(f'Recommended pathway : {pathway}')
print(f'Description         : {pathway_descriptions.get(pathway, "Unknown")}')

---
## 7. Phase-Folded Light Curve

In [ ]:
time_bjd = np.asarray(lc.time.jd, dtype=float)
flux     = np.asarray(lc.flux.value, dtype=float)

phase = ((time_bjd - signal.epoch_bjd) % signal.period_days) / signal.period_days
phase = np.where(phase > 0.5, phase - 1.0, phase)   # centre on transit

half_dur_phase = (signal.duration_hours / 24.0) / signal.period_days / 2.0

fig, ax = plt.subplots(figsize=(10, 4))
ax.scatter(phase, flux, s=1.0, color='steelblue', alpha=0.4, rasterized=True, label='Data')

# Shade transit window
ax.axvspan(-half_dur_phase, half_dur_phase, alpha=0.12, color='red', label='Transit window')
ax.axhline(1.0, color='gray', lw=0.8, ls='--')

ax.set_xlim(-0.15, 0.15)
ax.set_xlabel('Orbital phase', fontsize=11)
ax.set_ylabel('Normalised flux', fontsize=11)
ax.set_title(f'{TARGET_ID} — Phase-folded at P = {signal.period_days:.4f} d', fontsize=12)
ax.legend(fontsize=9, loc='lower right')

depth_label = signal.depth_ppm / 1e6
ax.annotate(f'Depth ≈ {signal.depth_ppm:.0f} ppm\nSNR = {signal.snr:.1f}',
            xy=(0, 1 - depth_label), xytext=(0.06, 1 - depth_label * 0.3),
            fontsize=9, color='red',
            arrowprops=dict(arrowstyle='->', color='red', lw=1.0))

plt.tight_layout()
plt.savefig('figures/02_phase_folded.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: figures/02_phase_folded.png')

---
## 8. Posterior Probability Bar Chart

In [ ]:
labels = ['Planet\ncandidate', 'Eclipsing\nbinary', 'Background\nEB',
          'Stellar\nvariability', 'Instrumental\nartifact', 'Known\nobject']
probs  = [posterior.planet_candidate, posterior.eclipsing_binary,
          posterior.background_eclipsing_binary, posterior.stellar_variability,
          posterior.instrumental_artifact, posterior.known_object]
colors = ['#2ecc71' if i == 0 else '#e74c3c' for i in range(6)]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.bar(labels, probs, color=colors, edgecolor='white', linewidth=0.8)

for bar, prob in zip(bars, probs):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f'{prob:.3f}', ha='center', va='bottom', fontsize=9)

ax.set_ylim(0, 1.05)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.set_ylabel('Posterior probability', fontsize=11)
ax.set_title(f'{TARGET_ID} — Hypothesis Posterior (P = {signal.period_days:.4f} d)', fontsize=12)
ax.axhline(0.5, color='gray', lw=0.8, ls='--', alpha=0.6)

plt.tight_layout()
plt.savefig('figures/03_posterior.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved: figures/03_posterior.png')

---
## 9. Candidate Report

In [ ]:
from IPython.display import Markdown, display
from datetime import datetime, timezone

# Estimate planet radius from transit depth and stellar radius
rp_rsun = STELLAR_RADIUS_RSUN * np.sqrt(signal.depth_ppm / 1e6)
rp_rearth = rp_rsun * 109.076   # R_sun / R_earth

report = f"""
## Candidate Report

**Generated**: {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M UTC')}  
**Pipeline**: exo_toolkit v0 (Bayesian log-score model)

---

### Signal Properties

| Parameter | Value |
|---|---|
| Target | {TARGET_ID} |
| Mission | {MISSION} |
| Candidate ID | {signal.candidate_id} |
| Period | {signal.period_days:.6f} d |
| Epoch (BJD) | {signal.epoch_bjd:.4f} |
| Duration | {signal.duration_hours:.2f} hr |
| Depth | {signal.depth_ppm:.0f} ppm |
| SNR | {signal.snr:.1f} |
| Transit count | {signal.transit_count} |
| Est. planet radius | {rp_rearth:.2f} R⊕ |

### Scoring Summary

| Score | Value |
|---|---|
| P(planet candidate) | {posterior.planet_candidate:.4f} |
| False positive probability | {scores.false_positive_probability:.4f} |
| Detection confidence | {scores.detection_confidence:.4f} |
| Novelty score | {scores.novelty_score:.4f} |
| Habitability interest | {scores.habitability_interest:.4f} |
| Follow-up value | {scores.followup_value:.4f} |
| Submission readiness | {scores.submission_readiness:.4f} |

### Recommended Pathway

**`{pathway}`** — {pathway_descriptions.get(pathway, '')}

---

> *This system identifies candidate signals only. No claim of planet confirmation is made.*
"""

display(Markdown(report))

---
## 10. All Signals Summary

Quick overview of all signals recovered from the iterative BLS search.

In [ ]:
if len(signals) > 1:
    fig, axes = plt.subplots(1, len(signals), figsize=(5 * len(signals), 4), sharey=True)
    if len(signals) == 1:
        axes = [axes]

    for ax, sig in zip(axes, signals):
        ph = ((time_bjd - sig.epoch_bjd) % sig.period_days) / sig.period_days
        ph = np.where(ph > 0.5, ph - 1.0, ph)
        hd = (sig.duration_hours / 24.0) / sig.period_days / 2.0

        ax.scatter(ph, flux, s=0.8, alpha=0.35, color='steelblue', rasterized=True)
        ax.axvspan(-hd, hd, alpha=0.12, color='red')
        ax.axhline(1.0, color='gray', lw=0.6, ls='--')
        ax.set_xlim(-0.15, 0.15)
        ax.set_title(f'P = {sig.period_days:.3f} d\nSNR = {sig.snr:.1f}', fontsize=9)
        ax.set_xlabel('Phase', fontsize=9)

    axes[0].set_ylabel('Normalised flux', fontsize=9)
    fig.suptitle(f'{TARGET_ID} — All recovered signals', fontsize=11, y=1.01)
    plt.tight_layout()
    plt.savefig('figures/04_all_signals.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Figure saved: figures/04_all_signals.png')
else:
    print('Only one signal recovered — skipping multi-signal plot.')

---

## References

- Gilbert, E. A., et al. "The First Habitable-Zone Earth-Sized Planet from TESS. I. Validation of the TOI-700 System." *AJ*, vol. 160, no. 3, 2020, p. 116. https://doi.org/10.3847/1538-3881/aba4b2
- Kovács, G., et al. "A Box-fitting Algorithm in the Search for Periodic Transits." *A&A*, vol. 391, 2002, pp. L23–L26.
- Ricker, G. R., et al. "Transiting Exoplanet Survey Satellite (TESS)." *JATIS*, vol. 1, 2015, p. 014003.
- Stassun, K. G., et al. "The TESS Input Catalog." *AJ*, vol. 156, 2018, p. 102.